# Configure Tests

In [ ]:
# =============================================================================
# SECTION 2: EVAL CONFIGURATION
# =============================================================================
# User-configurable parameters for the evaluation run.
# Adjust these before running the eval cells below.
# =============================================================================

# Path to the source CSV file (the structured ground truth data)
EVAL_CSV_PATH = "synthetic_biology_dataset.csv"

# Number of rows to evaluate
EVAL_N_ROWS = 2

# Starting row index (0-based offset into the CSV)
EVAL_OFFSET = 0

# Random seed for reproducibility
EVAL_RANDOM_SEED = 42

# Max retries per extraction attempt (passed to run_extraction_with_retry)
EVAL_MAX_RETRIES = 3

# Testing/ Evaluating General Structured Data Field Extaction
#### Extract structured-fields from unstructured input into a python dictionary


## Simple Minimal Data-Structured Task with Mistral API
This is a colab-notebook for testing data-extraction using a Mistral cloud model and cloud-api. This colab can run on any online device that runs a web browser (phone, tablet, laptop, desktop, etc.). This is a system for using the online cloud Mistral api, not the mistral models run locally in a local pipeline (e.g. using .gguf llama.cpp).

## Steps:
1. Configure api-key https://console.mistral.ai/api-keys/
2. Select Model https://docs.mistral.ai/getting-started/models/models_overview/
3. set "parameters" (or leave to default) https://docs.mistral.ai/capabilities/completion/sampling
3. Define what fields you want in a 'dictionary'
4. Describe those fields (e.g. their data-types, ranges, etc.)
5. Configure the extraction-task in the "Prompt"
6. Run all cells (Use the "Run all" button above)
7. (Optional) Download the session-log. Use the  folder-icon to the left <- to see the session log files; right click; download.




### Note
Key parts of the structured-data extraction process include
1. forcing a structure, pattern, or delimitor that can be detected in output
2. using the pattern or marker to extract the data
3. retrying in case of failure
4. retrying to check consistency and self-agreement (outlier check, sanity check, etc.)

Note: Escape-characters can become a problem with some formats (markdown-json is not always a viable format).

These can very significantly between models and sizes of models.

# Set Raw Input

In [ ]:
raw_text_blob = """
This is a cat! It is 23 cm tall. It gets 426 grams of food each day and weighs 9 kg! It is orange, and it was born on 1923-09-23. It is 3 years old! It can't swim. It can_ fly. It likes to watch youtube. It has 5 friends and has a popularity_score of 9.
"""

# Define Output Structure (Python Dictionary)
e.g.


```
"animal_type": "",
"weight_kg": "",
"height_cm": "",
"age_years": "",
"number_of_friends": "",
"birth_date": "",
"birth_unix": "",
"color": "",
"can_fly": "",
"can_swim": "",
"can_run": "",
"watches_youtube": "",
"daily_food_grams": "",
"popularity_score": "",
"```

In [ ]:
# this gets added to prompt
# only the keys from this are ever used
# (python does not allow type-only dicts, so dummy values are included)
target_schema_model_dict = {
    "animal_type": "", # string, categorical
    "weight_kg": 0.0, # float
    "height_cm": 0.0, # float
    "age_years": 0, # int
    "number_of_friends": 0, # int
    "birth_date": "", # str YYYY-MM-DD (datetime compatible)
    "birth_unix": 0, # int
    "color": "", # string, categorical
    "can_fly": False, # boolean
    "can_swim": False, # boolean
    "can_run": False, # boolean
    "watches_youtube": False, # boolean
    "daily_food_grams": 0.0, # float
    "popularity_score": 0.0, # float (?)
}

# Describe the Data Schema
- data types
- what the meaning and context of the field is

(note: make field names very clear)

e.g.
```
# this gets added to prompt
required fields and types, with example dummy values = {
    "animal_type":  # string, categorical
    "weight_kg": 0 # float
    "height_cm": 0 # float
    "age_years":  # int
    "number_of_friends":  # int
    "birth_date":  # str YYYY-MM-DD (datetime compatible)
    "birth_unix": # int
    "color": # string, categorical
    "can_fly":  # boolean
    "can_swim":  # boolean
    "can_run":  # boolean
    "watches_youtube":  # boolean
    "daily_food_grams":  # float
    "popularity_score":  # float
}
}
"""
```


In [ ]:
# this gets added to prompt
data_schema_doc_blurb = """
# this gets added to prompt
required fields and types, with example dummy values = {
    "animal_type":  # string, categorical
    "weight_kg": 0 # float
    "height_cm": 0 # float
    "age_years":  # int
    "number_of_friends":  # int
    "birth_date":  # str YYYY-MM-DD (datetime compatible)
    "birth_unix": # int
    "color": # string, categorical
    "can_fly":  # boolean
    "can_swim":  # boolean
    "can_run":  # boolean
    "watches_youtube":  # boolean
    "daily_food_grams":  # float
    "popularity_score":  # float
}
"""

# Set 'system prompt'

In [ ]:
def build_extraction_prompt(
    raw_text_input,
    data_schema,
    description_of_data_schema,
):

    full_prompt = f"""

    The task is to extract fields of structured data from an unstructured string.

    The fields and schema are:
    {data_schema}

    Descriptions of this including datatypes:
    {description_of_data_schema}

    The original unstructured text is:
    {raw_text_input}

    Output must be in correct markdown json format
    starting with ```json
    ending with ```

    Return a structured json object in markdown format,
    No other comments or output.
    """

    return full_prompt


In [ ]:
# set extraction_prompt_string
extraction_prompt_string = build_extraction_prompt(
    raw_text_blob,
    target_schema_model_dict,
    data_schema_doc_blurb,
)

# The Core API call:


- Select style-prompt to experiment with personality of answer (optional)
  - Modify the personality = "" text to describe the personality you want.
  - Specifying the language of reply can be done in the sytem-prompt

## Notes:

# Mistral models and names are updated and changes fairly often, check web for current.
Older models will hopefully be available somewhere online if not huggingface.

### Note: Colabs are slower
Free colabs are amazing for easily sharing and running code in a portable way,
but they are slower. Code in production, or run locally, will be faster than a colab.

#### From Mistral docs, See:
- https://docs.mistral.ai/
- https://docs.mistral.ai/getting-started/models/models_overview/
- https://docs.mistral.ai/platform/client/

```
curl --location "https://api.mistral.ai/v1/chat/completions" \
     --header 'Content-Type: application/json' \
     --header 'Accept: application/json' \
     --header "Authorization: Bearer $MISTRAL_API_KEY" \
     --data '{
    "model": "mistral-small-latest",
    "messages": [{"role": "user", "content": "Who is the most renowned French painter?"}]
  }'
```


## mistral-small-latest = Mixtral8x7
https://mistral.ai/news/mixtral-of-experts/

### response = requests.post(endpoint_url, headers=headers, json=request_body)


# Imports

In [ ]:
from datetime import datetime
import json
import re
import requests

# login

In [ ]:
"""
.env: get your environment variables:
  Using the Google Secretes (like.env) system
  built into colab on the left menu: the 'key' icon.
"""
from google.colab import userdata
mistral_api_key = userdata.get('mistral_api_key')


"""
Python Dot-env
"""
# from dotenv import load_dotenv
# import os

# load_dotenv()
# api_key = os.getenv("mistral_api_key")


"""
Hard Code (not the best idea)
"""
# mistral_api_key = 'xxx'

'\nHard Code (not the best idea)\n'

# Setup

Comment out the model you don't want to use.

In [ ]:
# Select Model
"""
https://docs.mistral.ai/api/

open-mistral-7b

open-mixtral-8x22b
open-mixtral-8x22b-2404

codestral-latest
codestral-2405


open-mistral-7b
(aka mistral-tiny-2312)
renamed from mistral-tiny
The endpoint mistral-tiny will be deprecated


Feb. 26, 2024

API endpoints: We renamed 3 API endpoints and added 2 model endpoints.

open-mistral-7b (aka mistral-tiny-2312): renamed from mistral-tiny. The endpoint mistral-tiny will be deprecated in three months.
open-mixtral-8x7B (aka mistral-small-2312): renamed from mistral-small. The endpoint mistral-small will be deprecated in three months.
mistral-small-latest (aka mistral-small-2402): new model.
mistral-medium-latest (aka mistral-medium-2312): old model. The previous mistral-medium has been dated and tagged as mistral-medium-2312. The endpoint mistral-medium will be deprecated in three months.
mistral-large-latest (aka mistral-large-2402): our new flagship model with leading performance.

"""

##################
# Open Mistral 7b
##################
# previously "tiny"
use_this_model = "open-mistral-7b"


###################
# Open Mixtral 8x7
###################
# previously "small"
use_this_model = "open-mixtral-8x7B"


######################
# open mixtral 8x22b
######################
# ...was 'medium'?
use_this_model = "open-mixtral-8x22b"


#######################
# Small, Medium, Large  (no 'tiny')
#######################
use_this_model = "mistral-small-latest"
use_this_model = "mistral-medium-latest"
use_this_model = "mistral-large-latest"

##############
# Codestral
##############
use_this_model = "codestral-latest"


use_this_model = "open-mistral-7b"

# Paremeters
https://docs.mistral.ai/capabilities/completion/sampling

In [ ]:
TEMPERATURE = 0.8 # higher number is more ~creative/variable answer

"""
Higher number accepts larger range of options,
or options with lower probabilities.
0.5 looks only at options with the top 50% likelihoods.
"""
TOP_P = 0.5  # top % (as fraction) considered for tokens,

"""
Range: [-2, 2]
Default: 0
"""
PRESENCE_PENALTY = 0

FREQUENCY_PENALTY = 0


# Blurb Generation Code

In [ ]:

"""
csv_unstructured_blurb_generator_w_negatives_vN.py

Purpose:
--------
This script reads a structured CSV dataset containing animal records,
and produces an augmented copy with an additional column containing
unstructured natural language descriptions ("blurbs") of each record.

Intent:
-------
The goal is to create a dataset that contains BOTH structured tabular data
AND unstructured text descriptions of the same information. This augmented
dataset can then be used to test and compare vector-database analytics
capabilities against known ground-truth structured data.

By deterministically generating text blurbs from known structured fields,
we create a controlled environment where:
- We KNOW what information is in each blurb
- We can verify if vector search finds the correct documents
- We can test corpus analytics questions with known correct answers

Input:
------
A CSV file with the following fields:
    animal_type, weight_kg, height_cm, age_years, number_of_friends,
    birth_date, birth_unix, color, can_fly, can_swim, can_run,
    watches_youtube, daily_food_grams, popularity_score, social_media

Output:
-------
A new CSV file with all original columns plus one new column:
    unstructured_description

The unstructured_description contains a natural language blurb that
describes the structured data using one of three rotating templates.

Design Decisions:
-----------------
- Template selection is deterministic: row_index % 3
- Float values are rounded to 3 decimal places in blurbs
- Boolean fields are converted to natural language phrases
- Original data is NOT modified; a new file is created
- No randomness; output is fully reproducible

Author: Synthetic Biology Datasets Project
"""

import csv
import traceback
import os
import sys


def round_float_to_three_decimals(value_to_round: float) -> float:
    """
    Round a floating point number to exactly 3 decimal places.

    Intent:
    -------
    Float values in the source CSV often have many decimal places
    (e.g., 28.602669690571616). For human-readable blurbs, we want
    cleaner numbers (e.g., 28.603). Three decimal places provides
    sufficient precision while maintaining readability.

    Parameters:
    -----------
    value_to_round : float
        The floating point number to be rounded.

    Returns:
    --------
    float
        The input value rounded to 3 decimal places.

    Example:
    --------
    >>> round_float_to_three_decimals(28.602669690571616)
    28.603
    """
    rounded_value = round(value_to_round, 3)
    return rounded_value


def convert_boolean_string_to_python_bool(boolean_string: str) -> bool:
    """
    Convert a string representation of a boolean to a Python bool.

    Intent:
    -------
    CSV files store all values as strings. When we read "True" or "False"
    from a CSV, we get the string "True", not the Python boolean True.
    This function handles that conversion safely.

    Parameters:
    -----------
    boolean_string : str
        A string that should be either "True" or "False" (case-sensitive
        as produced by Python's str(bool) conversion).

    Returns:
    --------
    bool
        True if the string is "True", False otherwise.

    Notes:
    ------
    This function treats any value that is not exactly "True" as False.
    This is a deliberate safety choice: unknown values default to False
    rather than raising an exception, making the pipeline more robust
    to minor data variations.
    """
    if boolean_string == "True":
        return True
    else:
        return False


def convert_can_ability_bool_to_phrase(can_do_ability: bool) -> str:
    """
    Convert a boolean ability flag to a natural language phrase.

    Intent:
    -------
    For abilities like "can_fly", "can_swim", "can_run", we want to
    produce readable phrases like "can" or "cannot" for use in
    natural language blurbs.

    Parameters:
    -----------
    can_do_ability : bool
        True if the animal has this ability, False otherwise.

    Returns:
    --------
    str
        "can" if True, "cannot" if False.

    Example:
    --------
    >>> convert_can_ability_bool_to_phrase(True)
    "can"
    >>> convert_can_ability_bool_to_phrase(False)
    "cannot"
    """
    if can_do_ability:
        return "can"
    else:
        return "cannot"


def convert_watches_youtube_bool_to_phrase(watches_youtube: bool) -> str:
    """
    Convert the watches_youtube boolean to a natural language phrase.

    Intent:
    -------
    The "watches_youtube" field needs a slightly different phrasing
    than abilities. We use "likes to" / "does not like to" to make
    grammatically correct sentences like "It likes to watch youtube."

    Parameters:
    -----------
    watches_youtube : bool
        True if the animal watches youtube, False otherwise.

    Returns:
    --------
    str
        "likes to" if True, "does not like to" if False.

    Example:
    --------
    >>> convert_watches_youtube_bool_to_phrase(True)
    "likes to"
    >>> convert_watches_youtube_bool_to_phrase(False)
    "does not like to"
    """
    if watches_youtube:
        return "likes to"
    else:
        return "does not like to"


def select_template_index_for_row(row_index: int) -> int:
    """
    Deterministically select which template (0, 1, or 2) to use for a row.

    Intent:
    -------
    We want variety in the blurb formats (to make vector search more
    realistic/challenging), but we also want reproducibility (so tests
    are repeatable). Using modulo arithmetic on the row index achieves
    both goals: the template varies by row, but the same row always
    gets the same template.

    Parameters:
    -----------
    row_index : int
        The zero-based index of the current row in the dataset.

    Returns:
    --------
    int
        An integer 0, 1, or 2 indicating which template to use.

    Example:
    --------
    >>> select_template_index_for_row(0)
    0
    >>> select_template_index_for_row(1)
    1
    >>> select_template_index_for_row(2)
    2
    >>> select_template_index_for_row(3)
    0
    """
    template_index = row_index % 3
    return template_index


def generate_blurb_using_template_zero(
    animal_type: str,
    weight_kg_rounded: float,
    height_cm_rounded: float,
    age_years: int,
    number_of_friends: int,
    birth_date: str,
    color: str,
    can_fly_phrase: str,
    can_swim_phrase: str,
    can_run_phrase: str,
    watches_youtube_phrase: str,
    daily_food_grams_rounded: float,
    popularity_score_rounded: float
) -> str:
    """
    Generate an unstructured text blurb using template format 0.

    Intent:
    -------
    This is the first of three template formats. It presents information
    in a specific order starting with animal type, then food, age, friends,
    physical characteristics, and abilities.

    Parameters:
    -----------
    animal_type : str
        The type of animal (e.g., "cat", "dog", "bird").
    weight_kg_rounded : float
        The animal's weight in kg, rounded to 3 decimal places.
    height_cm_rounded : float
        The animal's height in cm, rounded to 3 decimal places.
    age_years : int
        The animal's age in years.
    number_of_friends : int
        How many friends the animal has.
    birth_date : str
        The animal's birth date as a string (e.g., "2021-01-25").
    color : str
        The animal's color.
    can_fly_phrase : str
        "can" or "cannot" for flying ability.
    can_swim_phrase : str
        "can" or "cannot" for swimming ability.
    watches_youtube_phrase : str
        "likes to" or "does not like to" for youtube watching.
    daily_food_grams_rounded : float
        Daily food allowance in grams, rounded to 3 decimal places.
    popularity_score_rounded : float
        Popularity score, rounded to 3 decimal places.

    Returns:
    --------
    str
        A natural language blurb describing the animal.
    """
    blurb_text = (
        f"This is about a {animal_type}. "
        f"It gets a daily allowance of {daily_food_grams_rounded} grams of food. "
        f"It is {age_years} years old, and has {number_of_friends} friends. "
        f"This {animal_type} weighs {weight_kg_rounded} kg, and is {height_cm_rounded} cm tall. "
        f"It has a popularity_score of {popularity_score_rounded}. "
        f"It was born on {birth_date}, and is {color}. "
        f"It {can_fly_phrase} fly, and {can_swim_phrase} swim, and {can_run_phrase} run. "
        f"It {watches_youtube_phrase} watch youtube."
    )
    return blurb_text


def generate_blurb_using_template_one(
    animal_type: str,
    weight_kg_rounded: float,
    height_cm_rounded: float,
    age_years: int,
    number_of_friends: int,
    birth_date: str,
    color: str,
    can_fly_phrase: str,
    can_swim_phrase: str,
    can_run_phrase: str,
    watches_youtube_phrase: str,
    daily_food_grams_rounded: float,
    popularity_score_rounded: float
) -> str:
    """
    Generate an unstructured text blurb using template format 1.

    Intent:
    -------
    This is the second of three template formats. It presents information
    in a different order than template 0, and uses exclamation marks for
    some ability statements to create textual variety.

    Parameters:
    -----------
    (Same as generate_blurb_using_template_zero)

    Returns:
    --------
    str
        A natural language blurb describing the animal.
    """
    blurb_text = (
        f"This {animal_type} weighs {weight_kg_rounded} kg, and is {height_cm_rounded} cm tall. "
        f"It is {color} and was born on {birth_date}. "
        f"It gets {daily_food_grams_rounded} grams of food each day. "
        f"It is {age_years} years old. "
        f"It {can_fly_phrase} fly! "
        f"It {can_swim_phrase} swim! "
        f"It {can_run_phrase} run! "
        f"This {animal_type} has {number_of_friends} friends and has a popularity_score of {popularity_score_rounded}. "
        f"It {watches_youtube_phrase} watch youtube."
    )
    return blurb_text


def generate_blurb_using_template_two(
    animal_type: str,
    weight_kg_rounded: float,
    height_cm_rounded: float,
    age_years: int,
    number_of_friends: int,
    birth_date: str,
    color: str,
    can_fly_phrase: str,
    can_swim_phrase: str,
    can_run_phrase: str,
    watches_youtube_phrase: str,
    daily_food_grams_rounded: float,
    popularity_score_rounded: float
) -> str:
    """
    Generate an unstructured text blurb using template format 2.

    Intent:
    -------
    This is the third of three template formats. It presents information
    in yet another order, with different punctuation patterns (exclamation
    at the start, periods for abilities in the middle).

    Parameters:
    -----------
    (Same as generate_blurb_using_template_zero)

    Returns:
    --------
    str
        A natural language blurb describing the animal.
    """
    blurb_text = (
        f"This is a {animal_type}! "
        f"It is {height_cm_rounded} cm tall. "
        f"It gets {daily_food_grams_rounded} grams of food each day and weighs {weight_kg_rounded} kg! "
        f"It is {color}, and it was born on {birth_date}. "
        f"It {can_run_phrase} run. "
        f"It is {age_years} years old! "
        f"It {can_swim_phrase} swim. "
        f"It {can_fly_phrase} fly. "
        f"It {watches_youtube_phrase} watch youtube. "
        f"It has {number_of_friends} friends and has a popularity_score of {popularity_score_rounded}."
    )
    return blurb_text


def generate_blurb_for_row(row_data: dict, row_index: int) -> str:
    """
    Generate an unstructured text blurb for a single row of data.

    Intent:
    -------
    This function is the main orchestrator for blurb generation. It:
    1. Extracts and converts all necessary fields from the row
    2. Rounds float values to 3 decimal places
    3. Converts boolean strings to natural language phrases
    4. Selects the appropriate template based on row index
    5. Calls the appropriate template function

    This function encapsulates all the data transformation logic,
    keeping the main processing loop clean and simple.

    Parameters:
    -----------
    row_data : dict
        A dictionary containing one row of CSV data, where keys are
        column names and values are string representations of the data.
    row_index : int
        The zero-based index of this row in the dataset.

    Returns:
    --------
    str
        A natural language blurb describing the animal in this row.

    Raises:
    -------
    ValueError
        If required fields are missing or cannot be converted.
    KeyError
        If expected column names are not present in row_data.
    """
    # Extract and convert string values to appropriate types
    # Note: CSV reader returns all values as strings

    animal_type = row_data["animal_type"]
    color = row_data["color"]
    birth_date = row_data["birth_date"]

    # Convert numeric strings to numbers, then round floats
    weight_kg_raw = float(row_data["weight_kg"])
    weight_kg_rounded = round_float_to_three_decimals(weight_kg_raw)

    height_cm_raw = float(row_data["height_cm"])
    height_cm_rounded = round_float_to_three_decimals(height_cm_raw)

    daily_food_grams_raw = float(row_data["daily_food_grams"])
    daily_food_grams_rounded = round_float_to_three_decimals(daily_food_grams_raw)

    popularity_score_raw = float(row_data["popularity_score"])
    popularity_score_rounded = round_float_to_three_decimals(popularity_score_raw)

    # Integer fields
    age_years = int(row_data["age_years"])
    number_of_friends = int(row_data["number_of_friends"])

    # Convert boolean strings to Python bools, then to phrases
    can_fly_bool = convert_boolean_string_to_python_bool(row_data["can_fly"])
    can_fly_phrase = convert_can_ability_bool_to_phrase(can_fly_bool)

    can_swim_bool = convert_boolean_string_to_python_bool(row_data["can_swim"])
    can_swim_phrase = convert_can_ability_bool_to_phrase(can_swim_bool)

    can_run_bool = convert_boolean_string_to_python_bool(row_data["can_run"])
    can_run_phrase = convert_can_ability_bool_to_phrase(can_run_bool)

    watches_youtube_bool = convert_boolean_string_to_python_bool(row_data["watches_youtube"])
    watches_youtube_phrase = convert_watches_youtube_bool_to_phrase(watches_youtube_bool)

    # Select template based on row index
    template_index = select_template_index_for_row(row_index)

    # Generate blurb using the selected template
    if template_index == 0:
        generated_blurb = generate_blurb_using_template_zero(
            animal_type=animal_type,
            weight_kg_rounded=weight_kg_rounded,
            height_cm_rounded=height_cm_rounded,
            age_years=age_years,
            number_of_friends=number_of_friends,
            birth_date=birth_date,
            color=color,
            can_fly_phrase=can_fly_phrase,
            can_swim_phrase=can_swim_phrase,
            can_run_phrase=can_run_phrase,
            watches_youtube_phrase=watches_youtube_phrase,
            daily_food_grams_rounded=daily_food_grams_rounded,
            popularity_score_rounded=popularity_score_rounded
        )
    elif template_index == 1:
        generated_blurb = generate_blurb_using_template_one(
            animal_type=animal_type,
            weight_kg_rounded=weight_kg_rounded,
            height_cm_rounded=height_cm_rounded,
            age_years=age_years,
            number_of_friends=number_of_friends,
            birth_date=birth_date,
            color=color,
            can_fly_phrase=can_fly_phrase,
            can_swim_phrase=can_swim_phrase,
            can_run_phrase=can_run_phrase,
            watches_youtube_phrase=watches_youtube_phrase,
            daily_food_grams_rounded=daily_food_grams_rounded,
            popularity_score_rounded=popularity_score_rounded
        )
    else:
        generated_blurb = generate_blurb_using_template_two(
            animal_type=animal_type,
            weight_kg_rounded=weight_kg_rounded,
            height_cm_rounded=height_cm_rounded,
            age_years=age_years,
            number_of_friends=number_of_friends,
            birth_date=birth_date,
            color=color,
            can_fly_phrase=can_fly_phrase,
            can_swim_phrase=can_swim_phrase,
            can_run_phrase=can_run_phrase,
            watches_youtube_phrase=watches_youtube_phrase,
            daily_food_grams_rounded=daily_food_grams_rounded,
            popularity_score_rounded=popularity_score_rounded
        )

    return generated_blurb


def read_csv_file_to_list_of_dicts(input_file_path: str) -> tuple[list[dict], list[str]]:
    """
    Read a CSV file and return its contents as a list of dictionaries.

    Intent:
    -------
    This function handles the file I/O for reading the input CSV.
    It uses csv.DictReader which automatically uses the first row
    as column headers and returns each subsequent row as a dictionary.

    We also return the fieldnames separately so we can preserve
    column order when writing the output file.

    Parameters:
    -----------
    input_file_path : str
        The path to the input CSV file.

    Returns:
    --------
    tuple[list[dict], list[str]]
        A tuple containing:
        - A list of dictionaries, one per row
        - A list of field names (column headers) in original order

    Raises:
    -------
    FileNotFoundError
        If the input file does not exist.
    PermissionError
        If the file cannot be read due to permissions.
    ValueError
        If the CSV file is empty or has no header row.
    """
    list_of_row_dicts = []
    fieldnames_list = []

    with open(input_file_path, mode='r', encoding='utf-8', newline='') as csv_file_handle:
        csv_dict_reader = csv.DictReader(csv_file_handle)

        # Read all rows into memory as dictionaries
        for row_dict in csv_dict_reader:
            list_of_row_dicts.append(row_dict)

        # Capture the fieldnames from the reader AFTER reading rows
        # This ensures fieldnames is populated (it's None before first read)
        reader_fieldnames = csv_dict_reader.fieldnames

        # Explicit check for None to satisfy type checker and handle edge case
        if reader_fieldnames is None:
            raise ValueError(
                f"CSV file appears to be empty or has no header row: {input_file_path}"
            )

        fieldnames_list = list(reader_fieldnames)

    return list_of_row_dicts, fieldnames_list


# def write_augmented_data_to_csv(
#     output_file_path: str,
#     augmented_rows: list[dict],
#     output_fieldnames: list[str]
# ) -> None:
#     """
#     Write the augmented data to a new CSV file.

#     Intent:
#     -------
#     This function handles the file I/O for writing the output CSV.
#     It takes the list of augmented row dictionaries (which now include
#     the unstructured_description column) and writes them to a new file.

#     The original file is never modified; we always write to a new file.

#     Parameters:
#     -----------
#     output_file_path : str
#         The path where the output CSV file should be written.
#     augmented_rows : list[dict]
#         A list of dictionaries, one per row, including the new
#         unstructured_description field.
#     output_fieldnames : list[str]
#         The list of column names in the order they should appear
#         in the output file.

#     Returns:
#     --------
#     None

#     Raises:
#     -------
#     PermissionError
#         If the output file cannot be written due to permissions.
#     OSError
#         If there are other I/O errors (disk full, etc.).
#     """
#     with open(output_file_path, mode='w', encoding='utf-8', newline='') as csv_file_handle:
#         csv_dict_writer = csv.DictWriter(
#             csv_file_handle,
#             fieldnames=output_fieldnames
#         )

#         # Write the header row
#         csv_dict_writer.writeheader()

#         # Write all data rows
#         for row_dict in augmented_rows:
#             csv_dict_writer.writerow(row_dict)


# def generate_output_file_path(input_file_path: str) -> str:
#     """
#     Generate the output file path based on the input file path.

#     Intent:
#     -------
#     We want a predictable, non-destructive naming convention for output
#     files. This function takes the input path and produces an output
#     path with "_augmented" inserted before the file extension.

#     Example: "dataset.csv" -> "dataset_augmented.csv"
#     Example: "data/my_file.csv" -> "data/my_file_augmented.csv"

#     Parameters:
#     -----------
#     input_file_path : str
#         The path to the input CSV file.

#     Returns:
#     --------
#     str
#         The generated output file path.
#     """
#     # Split the path into directory, filename, and extension
#     directory_path = os.path.dirname(input_file_path)
#     filename_with_extension = os.path.basename(input_file_path)

#     # Split filename and extension
#     filename_without_extension, file_extension = os.path.splitext(filename_with_extension)

#     # Construct the new filename
#     augmented_filename = f"{filename_without_extension}_augmented{file_extension}"

#     # Reconstruct the full path
#     if directory_path:
#         output_file_path = os.path.join(directory_path, augmented_filename)
#     else:
#         output_file_path = augmented_filename

#     return output_file_path


# def process_csv_and_add_blurbs(input_file_path: str, output_file_path: str) -> int:
#     """
#     Main processing function: read CSV, add blurbs, write augmented CSV.

#     Intent:
#     -------
#     This is the main orchestration function that coordinates the entire
#     augmentation process:
#     1. Read the input CSV
#     2. For each row, generate a blurb
#     3. Add the blurb to the row data
#     4. Write all augmented rows to the output CSV

#     This function is separate from main() to allow for easier testing
#     and reuse. It contains the core business logic without command-line
#     argument handling.

#     Parameters:
#     -----------
#     input_file_path : str
#         The path to the input CSV file.
#     output_file_path : str
#         The path where the augmented CSV should be written.

#     Returns:
#     --------
#     int
#         The number of rows processed.

#     Raises:
#     -------
#     FileNotFoundError
#         If the input file does not exist.
#     ValueError
#         If the CSV data cannot be processed (missing fields, bad values).
#     """
#     # Read the input CSV
#     print(f"Reading input file: {input_file_path}")
#     rows_list, original_fieldnames = read_csv_file_to_list_of_dicts(input_file_path)
#     print(f"Found {len(rows_list)} rows to process")

#     # Prepare the output fieldnames (original plus new column)
#     new_column_name = "unstructured_description"
#     output_fieldnames = original_fieldnames + [new_column_name]

#     # Process each row: generate blurb and add to row dict
#     augmented_rows_list = []

#     """
#     To prevent status prints from being too abundant
#     as data files are typically not of piceune quantity,
#     getting list-length//10, and using:
#         if (row_index + 1) % short_status_base == 0:
#     should give usually 10 status prints, also fitting
#     within 80x24 terminal size.
#     """
#     short_status_base = len(rows_list)//10

#     for row_index, row_dict in enumerate(rows_list):
#         # Generate the blurb for this row
#         blurb_text = generate_blurb_for_row(row_dict, row_index)

#         # Create a new dict with the blurb added (don't modify original)
#         augmented_row_dict = dict(row_dict)
#         augmented_row_dict[new_column_name] = blurb_text

#         augmented_rows_list.append(augmented_row_dict)

#         # Progress indicator for large files
#         if (row_index + 1) % short_status_base == 0:
#             print(f"  Processed {row_index + 1} rows...")

#     # Write the augmented data to the output file
#     print(f"Writing output file: {output_file_path}")
#     write_augmented_data_to_csv(output_file_path, augmented_rows_list, output_fieldnames)

#     rows_processed_count = len(augmented_rows_list)
#     print(f"Successfully processed {rows_processed_count} rows")

#     return rows_processed_count


# def main() -> None:
#     """
#     Entry point for the CSV summary blurb augmentor script.

#     Intent:
#     -------
#     This function handles:
#     1. Command-line argument parsing
#     2. Input validation
#     3. Calling the main processing function
#     4. Error handling and user feedback

#     Usage:
#     ------
#     python csv_summary_blurb_augmentor.py <input_csv_path> [output_csv_path]

#     If output_csv_path is not provided, it will be auto-generated by
#     adding "_augmented" to the input filename.

#     Examples:
#     ---------
#     python csv_summary_blurb_augmentor.py dataset.csv
#     python csv_summary_blurb_augmentor.py data/input.csv data/output.csv
#     """
#     try:
#         # Parse command-line arguments
#         if len(sys.argv) < 2:
#             print("Usage: python csv_summary_blurb_augmentor.py <input_csv_path> [output_csv_path]")
#             print("")
#             print("Arguments:")
#             print("  input_csv_path   - Path to the input CSV file (required)")
#             print("  output_csv_path  - Path for the output CSV file (optional)")
#             print("")
#             print("If output_csv_path is not provided, the output will be named")
#             print("<input_filename>_augmented.csv in the same directory.")
#             sys.exit(1)

#         input_csv_path = sys.argv[1]

#         # Determine output path (from argument or auto-generated)
#         if len(sys.argv) >= 3:
#             output_csv_path = sys.argv[2]
#         else:
#             output_csv_path = generate_output_file_path(input_csv_path)

#         # Validate input file exists
#         if not os.path.exists(input_csv_path):
#             print(f"ERROR: Input file not found: {input_csv_path}")
#             sys.exit(1)

#         # Check that we're not about to overwrite the input file
#         input_absolute_path = os.path.abspath(input_csv_path)
#         output_absolute_path = os.path.abspath(output_csv_path)

#         if input_absolute_path == output_absolute_path:
#             print("ERROR: Output path is the same as input path.")
#             print("This script does not modify files in place.")
#             print("Please specify a different output path.")
#             sys.exit(1)

#         # Run the main processing
#         print("=" * 60)
#         print("CSV Summary Blurb Augmentor")
#         print("=" * 60)

#         rows_processed = process_csv_and_add_blurbs(input_csv_path, output_csv_path)

#         print("=" * 60)
#         print(f"Complete! Processed {rows_processed} rows.")
#         print(f"Output written to: {output_csv_path}")
#         print("=" * 60)

#     except FileNotFoundError as file_not_found_error:
#         print(f"ERROR: File not found - {file_not_found_error}")
#         traceback.print_exc()
#         sys.exit(1)

#     except PermissionError as permission_error:
#         print(f"ERROR: Permission denied - {permission_error}")
#         traceback.print_exc()
#         sys.exit(1)

#     except KeyError as key_error:
#         print(f"ERROR: Missing expected column in CSV - {key_error}")
#         print("Please ensure the input CSV has all required columns.")
#         traceback.print_exc()
#         sys.exit(1)

#     except ValueError as value_error:
#         print(f"ERROR: Invalid data value - {value_error}")
#         print("Please ensure all numeric fields contain valid numbers.")
#         traceback.print_exc()
#         sys.exit(1)

#     except Exception as unexpected_error:
#         print(f"ERROR: An unexpected error occurred - {unexpected_error}")
#         traceback.print_exc()
#         sys.exit(1)


# if __name__ == "__main__":
#     main()

# Fields Type Validation Code

In [ ]:
# -*- coding: utf-8 -*-
"""
Datatype Validation and Coercion Module

This module provides functions to validate and coerce values in an extracted
dictionary to their required datatypes. This is the step between key validation
and returning the final validated dict.

Pipeline position:
    extract JSON from markdown
        → validate keys match schema
            → [THIS MODULE] validate/coerce datatypes
                → return validated dict
"""

import re
import traceback


# =============================================================================
# SCHEMA DEFINITION
# =============================================================================

# Maps each field key to its required datatype.
# Supported type strings: "str", "int", "float", "bool", "date_str"
# All fields accept None/null/empty as valid (returns None).

FIELD_TYPE_SCHEMA = {
    "animal_type": "str",
    "weight_kg": "float",
    "height_cm": "float",
    "age_years": "int",
    "number_of_friends": "int",
    "birth_date": "date_str",  # special: must match YYYY-MM-DD
    "birth_unix": "int",
    "color": "str",
    "can_fly": "bool",
    "can_swim": "bool",
    "can_run": "bool",
    "watches_youtube": "bool",
    "daily_food_grams": "float",
    "popularity_score": "float",
}


# =============================================================================
# NULL/EMPTY DETECTION
# =============================================================================

def is_null_or_empty(value) -> bool:
    """
    Checks whether a value should be treated as null/empty.

    Null/empty values are acceptable for all fields and will be
    converted to Python None in the final output.

    Parameters:
    -----------
    value : any
        The value to check.

    Returns:
    --------
    bool
        True if the value is considered null/empty, False otherwise.

    Examples:
    ---------
    >>> is_null_or_empty(None)
    True
    >>> is_null_or_empty("")
    True
    >>> is_null_or_empty("null")
    True
    >>> is_null_or_empty(0)
    False
    >>> is_null_or_empty("hello")
    False
    """

    # Python None
    if value is None:
        return True

    # Empty string
    if value == "":
        return True

    # String representations of null (case-insensitive)
    if isinstance(value, str):
        value_lower = value.strip().lower()
        if value_lower in ("null", "none", "n/a", "na", ""):
            return True

    return False


# =============================================================================
# TYPE COERCION FUNCTIONS
# =============================================================================

def coerce_to_str(value, field_name: str) -> str:
    """
    Coerces a value to a string.

    Almost anything can become a string, so this is lenient.
    The main purpose is to ensure the final type is str.

    Parameters:
    -----------
    value : any
        The value to coerce.
    field_name : str
        The name of the field (for error messages).

    Returns:
    --------
    str
        The value as a string.

    Raises:
    -------
    ValueError
        If coercion fails (unlikely for str, but included for consistency).
    """

    try:
        return str(value)

    except Exception as e:
        raise ValueError(
            f"Field '{field_name}': Cannot coerce value '{value}' "
            f"(type: {type(value).__name__}) to str. Error: {e}"
        )


def coerce_to_int(value, field_name: str) -> int:
    """
    Coerces a value to an integer.

    Accepts:
    - int (pass through)
    - float (truncates toward zero)
    - str that represents a number (e.g., "42", "3.7" -> 3)

    Truncation from float to int is explicitly acceptable per requirements.

    Parameters:
    -----------
    value : any
        The value to coerce.
    field_name : str
        The name of the field (for error messages).

    Returns:
    --------
    int
        The value as an integer.

    Raises:
    -------
    ValueError
        If coercion fails.
    """

    try:
        # If already int, return directly
        if isinstance(value, int) and not isinstance(value, bool):
            return value

        # If float, truncate to int
        if isinstance(value, float):
            return int(value)

        # If string, try to parse as float first (handles "3.7"), then int
        if isinstance(value, str):
            # Strip whitespace
            value_stripped = value.strip()
            # Parse as float first to handle decimal strings
            float_value = float(value_stripped)
            return int(float_value)

        # Last resort: try direct int conversion
        return int(value)

    except Exception as e:
        raise ValueError(
            f"Field '{field_name}': Cannot coerce value '{value}' "
            f"(type: {type(value).__name__}) to int. Error: {e}"
        )


def coerce_to_float(value, field_name: str) -> float:
    """
    Coerces a value to a float.

    Accepts:
    - float (pass through)
    - int (promoted to float)
    - str that represents a number (e.g., "3.14", "42")

    Parameters:
    -----------
    value : any
        The value to coerce.
    field_name : str
        The name of the field (for error messages).

    Returns:
    --------
    float
        The value as a float.

    Raises:
    -------
    ValueError
        If coercion fails.
    """

    try:
        # If already float, return directly
        if isinstance(value, float):
            return value

        # If int (but not bool), promote to float
        if isinstance(value, int) and not isinstance(value, bool):
            return float(value)

        # If string, parse
        if isinstance(value, str):
            value_stripped = value.strip()
            return float(value_stripped)

        # Last resort
        return float(value)

    except Exception as e:
        raise ValueError(
            f"Field '{field_name}': Cannot coerce value '{value}' "
            f"(type: {type(value).__name__}) to float. Error: {e}"
        )


def coerce_to_bool(value, field_name: str) -> bool:
    """
    Coerces a value to a boolean.

    Accepts a wide variety of truthy/falsy representations:
    - bool: True, False (pass through)
    - int: 0 -> False, 1 -> True
    - str: "true", "false", "yes", "no", "1", "0", "t", "f", "y", "n"
           (case-insensitive)

    Parameters:
    -----------
    value : any
        The value to coerce.
    field_name : str
        The name of the field (for error messages).

    Returns:
    --------
    bool
        The value as a boolean.

    Raises:
    -------
    ValueError
        If coercion fails (value not recognizable as boolean).
    """

    # Already bool
    if isinstance(value, bool):
        return value

    # Integer: 0 or 1
    if isinstance(value, int):
        if value == 0:
            return False
        elif value == 1:
            return True
        else:
            raise ValueError(
                f"Field '{field_name}': Integer value '{value}' is not "
                f"0 or 1, cannot interpret as boolean."
            )

    # Float: 0.0 or 1.0
    if isinstance(value, float):
        if value == 0.0:
            return False
        elif value == 1.0:
            return True
        else:
            raise ValueError(
                f"Field '{field_name}': Float value '{value}' is not "
                f"0.0 or 1.0, cannot interpret as boolean."
            )

    # String representations
    if isinstance(value, str):
        value_lower = value.strip().lower()

        # Truthy strings
        if value_lower in ("true", "yes", "1", "t", "y"):
            return True

        # Falsy strings
        if value_lower in ("false", "no", "0", "f", "n"):
            return False

        raise ValueError(
            f"Field '{field_name}': String value '{value}' is not a "
            f"recognized boolean representation."
        )

    # Unrecognized type
    raise ValueError(
        f"Field '{field_name}': Cannot coerce value '{value}' "
        f"(type: {type(value).__name__}) to bool."
    )


def coerce_to_date_str(value, field_name: str) -> str:
    """
    Validates and returns a date string in YYYY-MM-DD format.

    This does NOT parse or validate that the date is a real calendar date.
    It only checks the string format matches the pattern.

    Parameters:
    -----------
    value : any
        The value to validate (expected to be a string).
    field_name : str
        The name of the field (for error messages).

    Returns:
    --------
    str
        The validated date string in YYYY-MM-DD format.

    Raises:
    -------
    ValueError
        If the value is not a string or does not match YYYY-MM-DD format.
    """

    # Regex pattern for YYYY-MM-DD
    # Requires exactly 4 digits, hyphen, 2 digits, hyphen, 2 digits
    DATE_PATTERN = r"^\d{4}-\d{2}-\d{2}$"

    # Must be a string
    if not isinstance(value, str):
        raise ValueError(
            f"Field '{field_name}': Expected date string, got "
            f"'{value}' (type: {type(value).__name__})."
        )

    # Strip whitespace
    value_stripped = value.strip()

    # Check pattern
    if not re.match(DATE_PATTERN, value_stripped):
        raise ValueError(
            f"Field '{field_name}': Date string '{value}' does not match "
            f"required format YYYY-MM-DD."
        )

    return value_stripped


# =============================================================================
# TYPE COERCION DISPATCHER
# =============================================================================

def coerce_value_to_type(
    value,
    required_type: str,
    field_name: str,
):
    """
    Dispatches to the appropriate coercion function based on required_type.

    This is the central routing function that maps type strings to
    their coercion functions.

    Parameters:
    -----------
    value : any
        The value to coerce.
    required_type : str
        One of: "str", "int", "float", "bool", "date_str"
    field_name : str
        The name of the field (for error messages).

    Returns:
    --------
    The coerced value in the correct type.

    Raises:
    -------
    ValueError
        If coercion fails.
    KeyError
        If required_type is not recognized.
    """

    # Dispatch table mapping type strings to coercion functions
    coercion_dispatch = {
        "str": coerce_to_str,
        "int": coerce_to_int,
        "float": coerce_to_float,
        "bool": coerce_to_bool,
        "date_str": coerce_to_date_str,
    }

    # Check if type is recognized
    if required_type not in coercion_dispatch:
        raise KeyError(
            f"Field '{field_name}': Unknown required type '{required_type}'. "
            f"Supported types: {list(coercion_dispatch.keys())}"
        )

    # Get coercion function and call it
    coercion_function = coercion_dispatch[required_type]
    return coercion_function(value, field_name)


# =============================================================================
# MAIN VALIDATION FUNCTION
# =============================================================================

def validate_and_coerce_extracted_dict(
    extracted_dict: dict,
    field_type_schema: dict = None,
) -> dict:
    """
    Validates and coerces all values in an extracted dictionary to their
    required datatypes.

    This function is called AFTER key validation has passed. It iterates
    through each key-value pair, checks for null/empty, and attempts to
    coerce to the required type.

    On success: Returns a new dict with all values coerced to correct types.
    On failure: Raises ValueError with details (triggers retry upstream).

    Parameters:
    -----------
    extracted_dict : dict
        The dictionary extracted from the LLM response, already validated
        for correct keys.
    field_type_schema : dict, optional
        A dict mapping field names to required type strings.
        If None, uses the module-level FIELD_TYPE_SCHEMA.

    Returns:
    --------
    dict
        A new dictionary with all values coerced to their required types.
        Null/empty values are converted to None.

    Raises:
    -------
    ValueError
        If any value cannot be coerced to its required type.
        The exception message includes field name, value, and expected type.

    Example:
    --------
    >>> raw = {"age_years": "25", "can_fly": "yes", "color": None}
    >>> validated = validate_and_coerce_extracted_dict(raw)
    >>> validated
    {"age_years": 25, "can_fly": True, "color": None}
    """

    # Use module-level schema if none provided
    if field_type_schema is None:
        field_type_schema = FIELD_TYPE_SCHEMA

    # Output dict to build
    validated_dict = {}

    # Track errors for comprehensive reporting
    # (We fail on first error, but this structure allows future batch reporting)
    errors_encountered = []

    # Iterate through each field in the schema
    for field_name, required_type in field_type_schema.items():

        # Get value from extracted dict
        # (Key existence already validated upstream, but defensive check)
        if field_name not in extracted_dict:
            raise ValueError(
                f"Field '{field_name}' missing from extracted dict. "
                f"This should have been caught by key validation."
            )

        raw_value = extracted_dict[field_name]

        # --- Check for null/empty ---
        if is_null_or_empty(raw_value):
            # Null/empty is acceptable for all fields
            validated_dict[field_name] = None
            continue

        # --- Attempt coercion ---
        try:
            coerced_value = coerce_value_to_type(
                value=raw_value,
                required_type=required_type,
                field_name=field_name,
            )
            validated_dict[field_name] = coerced_value

        except (ValueError, KeyError) as e:
            # Coercion failed - raise to trigger retry
            raise ValueError(
                f"Datatype validation failed for field '{field_name}'. "
                f"Raw value: '{raw_value}' (type: {type(raw_value).__name__}). "
                f"Required type: '{required_type}'. "
                f"Error: {e}"
            )

    return validated_dict


# =============================================================================
# INTEGRATION WRAPPER (for use in existing pipeline)
# =============================================================================

def validate_datatypes_or_fail(
    extracted_dict: dict,
    field_type_schema: dict = None,
) -> dict | bool:
    """
    Wrapper function that matches the pattern used in the existing pipeline.

    Calls validate_and_coerce_extracted_dict() and catches exceptions,
    returning False on failure (matching the pattern of
    extract_json_as_pydict_from_unstructured_text()).

    Use this for drop-in integration with the existing retry loop.

    Parameters:
    -----------
    extracted_dict : dict
        The dictionary to validate.
    field_type_schema : dict, optional
        Schema mapping field names to type strings.

    Returns:
    --------
    dict
        The validated and coerced dictionary on success.
    False
        On validation failure (triggers retry upstream).
    """

    try:
        validated_dict = validate_and_coerce_extracted_dict(
            extracted_dict=extracted_dict,
            field_type_schema=field_type_schema,
        )
        return validated_dict

    except ValueError as e:
        print(f"\nDATATYPE VALIDATION FAILED: {e}")
        print(traceback.format_exc())
        return False

    except Exception as e:
        print(f"\nUNEXPECTED ERROR IN DATATYPE VALIDATION: {e}")
        print(traceback.format_exc())
        return False

# API Code

In [ ]:
# import requests
# import json
# import os
# import re
# from google.colab import userdata

"""
# mistral_api_key = userdata.get('mistral_api_key')

# Define the endpoint URL
endpoint_url = "https://api.mistral.ai/v1/chat/completions"

# Set the headers
headers = {
  "Content-Type": "application/json",
  "Accept": "application/json",
  "Authorization": f"Bearer {mistral_api_key}"
}

# mode: [{"role": "user", "content": "say yes"}]

    # Define the request body
    request_body = {
      "model": "mistral-small-latest",
      "messages": [{"role": "user", "content": user_input}]
    }

    # Send the request
    response = requests.post(endpoint_url, headers=headers, json=request_body)
"""


def add_to_context_history(role, comment):

    if role == 'user':
        segment = {"role": "user", "content": comment}

    elif role == 'assistant':
        segment = {"role": "assistant", "content": comment}

    elif role == 'system':
        segment = {"role": "system", "content": comment}

    else:
        print("add_to_context_history(role, comment)")
        print(role, comment)
        print('error')

    return segment


def ask_mistral_api(context_history, use_this_model):


    # Define the endpoint URL
    endpoint_url = "https://api.mistral.ai/v1/chat/completions"

    # Set the headers
    headers = {
        "Content-Type": "application/json",
        "Accept": "application/json",
        "Authorization": f"Bearer {mistral_api_key}"
    }

    # Define the request body
    request_body = {
        "model": use_this_model,
        "messages": context_history,

        "temperature": TEMPERATURE,
        "top_p": TOP_P,
        "presence_penalty": PRESENCE_PENALTY,
        "frequency_penalty": FREQUENCY_PENALTY,
    }

    #################
    #################
    # Hit the ai api
    #################
    #################
    # Send the request
    response = requests.post(endpoint_url, headers=headers, json=request_body)

    # Check the response status code
    if response.status_code != 200:
        raise Exception(f"Error: {response.status_code} {response.text}")

    return response


def simple_ask_mistral_cloud(input_string, use_this_model):
    """
    you have: a string
    you need: a response

    1. make minimal history contexxt
    2. make a generic system instruction, for show
    3. make system-user context: string input
    4. ask mistral for that model
    5. extract just the response string
    6. return only reply (no 'history')
    """

    # 1. make minimal history contexxt
    context_history = []

    # 2. make a generic system instruction
    generic_system_instruction = "You are helpful and answer accurately."
    context_history.append( add_to_context_history("system", generic_system_instruction) )

    # 3. make system-user context: string input
    context_history.append( add_to_context_history("user", input_string) )

    # 4. ask mistral for that model
    response = ask_mistral_api(context_history, use_this_model)


    # Get the response data
    response_data = response.json()


    # 5. extract just the response string

    ##
    ##
    # Turn this print on to see full return data
    ##
    ##
    """
    e.g.
    {
      "id": "635cb8d445ujhe5546bb64e5e7",
      "object": "chat.completion",
      "created": 170hrjfjf7084,
      "model": "open-mistral-7b",
      "choices": [
        {
          "index": 0,
          "message": {
            "role": "assistant",
            "content": "Enjoy your cup of tea!"
          },
          "finish_reason": "stop",
          "logprobs": null
        }
      ],
      "usage": {
        "prompt_tokens": 575,
        "total_tokens": 629,
        "completion_tokens": 54
      }
    }
    """
    # print(json.dumps(response_data, indent=2))
    # print(type(response_data))

    output = response_data
    # print(type(output))
    # print(type(output["choices"][0]))

    # extract just the 'what they said' part out
    assistant_says = output["choices"][0]['message']['content']

    # 6. return only reply (no 'history')
    return assistant_says


# Extraction Code

In [ ]:
import json

# Helper Function
def extract_json_as_pydict_from_unstructured_text(dict_str, model_dict):
    """
    This function CAN fail and should fail
    if the AI needs to retry at a task.
    Do not stop server when this this triggers an exception.

    edge case: before there is a populated output_log

    if passing, this function will return a valid json object
    """

    """
    Extracts JSON string enclosed between ```json and ``` markers.

    Parameters:
    - text (str): The input text containing the JSON block.

    Returns:
    - str: The extracted JSON string, or an empty string if no JSON block is found.
    """
    print(f"\n\n Starting check_function_description_keys, dict_str -> {dict_str}")

    ########################
    # Check Json Formatting
    ########################
    try:
        pattern = r'```json\n([\s\S]*?)\n```'
        match = re.search(pattern, dict_str)
        dict_str =  match.group(1) if match else ''

    except Exception as e:
        print(f"\nTRY AGAIN: check_function_description_keys() extraction from markdown failed: {e}")
        print(f"Failed dict_str -> {dict_str}")
        return False

    print(f"\n extracted from markdown ->{dict_str}")

    # clean
    try:
        # try safety cleaning
        dict_str = dict_str.replace("True", "true")
        dict_str = dict_str.replace("False", "false")
        dict_str = dict_str.replace("None", "null")

        # # This conflicted with free language in description section...
        # dict_str = dict_str.replace("'", '"')

        # remove trailing delimiter comma
        print(f"{dict_str[:-6]}")
        dict_str = dict_str.replace('",\n}', '"\n}')

    except Exception as e:
        print(f"\nTRY AGAIN:try safety cleaning: {e}")
        print(f"Failed dict_str -> {dict_str}")
        return False

    # load
    try:
        # try converting
        dict_str = json.loads(dict_str)

    except Exception as e:
        print(f"\nTRY AGAIN: trying json.loads(dict_str) Dictionary load failed: {e}")
        print(f"Failed dict_str -> {dict_str}")
        return False


    # check if keys are the same
    try:
        result = dict_str.keys() == model_dict.keys()
        if result is False:
            print(f"Failed: keys are not the same.")
            print(f"Failed dict_str -> {dict_str}")
            return False

    except Exception as e:
        print(f"\nTRY AGAIN: Failed with Exception: keys are not the same: {e}")
        print(f"Failed dict_str -> {dict_str}")
        return False


    # --- Validate/Clean Datatypes ---
    validated_dict = validate_datatypes_or_fail(dict_str)
    if validated_dict is False:
        print(f"TRY AGAIN: Datatype validation failed.")
        return False

    # if ok...
    return dict_str



In [ ]:
# import json
import re
import traceback
# from datetime import datetime


def initialize_session_log(
    raw_text_blob: str,
    target_schema_model_dict: dict,
    use_this_model: str,
) -> str:
    """
    Creates a new timestamped session log file and writes
    the opening configuration block to it.

    Everything printed during the session will also be
    appended to this file via print_and_log().

    Parameters:
    -----------
    raw_text_blob : str
        The raw input text to be processed.
    target_schema_model_dict : dict
        The schema dict defining expected output fields.
    use_this_model : str
        The model name string used for this session.

    Returns:
    --------
    str
        The filepath of the created session log file.
    """

    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
    session_log_filepath = f"session_log_{timestamp}.txt"

    opening_block = (
        f"\n{'='*60}\n"
        f"SESSION LOG\n"
        f"{'='*60}\n"
        f"Timestamp      : {timestamp}\n"
        f"Model          : {use_this_model}\n"
        f"Schema         :\n{json.dumps(target_schema_model_dict, indent=2)}\n"
        f"Raw Input Blob :\n{raw_text_blob}\n"
        f"{'='*60}\n"
    )

    try:
        with open(session_log_filepath, 'w') as log_file:
            log_file.write(opening_block)
        print(opening_block)

    except Exception as e:
        print(f"WARNING: Could not create session log file: {e}")
        print(traceback.format_exc())

    return session_log_filepath


def print_and_log(
    message: str,
    session_log_filepath: str,
) -> None:
    """
    Prints a message to stdout AND appends it to the session log file.

    This is the single point of output throughout the pipeline.
    Nothing should be printed directly — always use this function
    so that stdout and the log file stay in sync.

    Parameters:
    -----------
    message : str
        The string to print and append to the log.
    session_log_filepath : str
        Path to the active session log file.

    Returns:
    --------
    None
    """

    print(message)

    try:
        with open(session_log_filepath, 'a') as log_file:
            log_file.write(message + "\n")

    except Exception as e:
        # Print warning to stdout only — do not recurse into print_and_log
        print(f"WARNING: Could not append to session log file: {e}")
        print(traceback.format_exc())


def save_extraction_results(
    raw_text_blob: str,
    target_schema_model_dict: dict,
    use_this_model: str,
    result_dict: dict | None,
    attempts_made: int,
    session_log_filepath: str,
) -> None:
    """
    Saves the final extraction results to a timestamped JSON file.

    Called on success (with the validated dict) and on total failure
    (with result_dict=None). Captures all inputs, settings, and output
    together in one file for reproducibility and inspection.

    Parameters:
    -----------
    raw_text_blob : str
        The original raw input text.
    target_schema_model_dict : dict
        The schema dict.
    use_this_model : str
        Model name string.
    result_dict : dict or None
        The validated extracted dict on success, or None on failure.
    attempts_made : int
        How many attempts were made before this result.
    session_log_filepath : str
        Path to the session log file (recorded here for cross-reference).

    Returns:
    --------
    None
    """

    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
    results_filepath = f"extraction_results_{timestamp}.json"

    results_payload = {
        "timestamp": timestamp,
        "model_used": use_this_model,
        "attempts_made": attempts_made,
        "raw_text_blob": raw_text_blob,
        "target_schema_model_dict": target_schema_model_dict,
        "extraction_result": result_dict,
        "session_log_filepath": session_log_filepath,
    }

    try:
        with open(results_filepath, 'w') as results_file:
            json.dump(results_payload, results_file, indent=2)

        print_and_log(
            f"\nResults saved to: {results_filepath}",
            session_log_filepath,
        )

    except Exception as e:
        print_and_log(
            f"WARNING: Could not save results file: {e}\n{traceback.format_exc()}",
            session_log_filepath,
        )


def run_extraction_with_retry(
    raw_text_blob: str,
    target_schema_model_dict: dict,
    extraction_prompt_string: str,
    use_this_model: str,
    max_retries: int = 3,
) -> dict | None:
    """
    Main extraction pipeline loop.

    Takes a fully pre-built prompt string (already assembled by
    build_extraction_prompt() before this function is called),
    sends it to the Mistral API, and attempts to extract a validated
    Python dict matching the target schema.

    Retries up to max_retries times on JSON extraction failure.

    All output — raw API responses, extraction attempts, successes,
    failures — is printed to stdout AND appended to a timestamped
    session log file via print_and_log().

    Final results (success or failure) are saved to a separate
    timestamped JSON file via save_extraction_results().

    Context history is constructed ONCE before the loop.
    The prompt does not change between retries.

    API-level exceptions (non-200 status from ask_mistral_api())
    are NOT caught here — they propagate up and stop execution.
    JSON extraction failures are caught and retried.

    Parameters:
    -----------
    raw_text_blob : str
        The original raw input text. Used for saving results.
        The prompt string is already built before this is called.
    target_schema_model_dict : dict
        The schema dict defining expected output fields and structure.
        Passed to extract_json_as_pydict_from_unstructured_text()
        for key validation.
    extraction_prompt_string : str
        The fully assembled prompt string from build_extraction_prompt().
    use_this_model : str
        The Mistral model name string.
    max_retries : int
        Maximum number of API call + extraction attempts. Default 3.

    Returns:
    --------
    dict
        Validated Python dict matching the schema on success.
    None
        If all retries are exhausted without a valid extraction.
    """

    # --- Initialize session log (opened once, appended throughout) ---
    session_log_filepath = initialize_session_log(
        raw_text_blob,
        target_schema_model_dict,
        use_this_model,
    )

    # ----------------------------------------------------------------
    # Construct context_history ONCE before the loop.
    # Same history is sent on every retry attempt.
    # ----------------------------------------------------------------
    context_history = []

    generic_system_instruction = (
        "You are a precise data extraction assistant. "
        "Follow the instructions exactly. "
        "Return only valid JSON in markdown format. "
        "No other text or commentary."
    )

    # system message
    context_history.append(
        add_to_context_history("system", generic_system_instruction)
    )

    # user message: the full pre-built prompt
    context_history.append(
        add_to_context_history("user", extraction_prompt_string)
    )

    print_and_log(
        (
            f"\nContext history constructed."
            f"\nStarting extraction loop."
            f"\nMax retries: {max_retries}\n"
        ),
        session_log_filepath,
    )

    # ----------------------------------------------------------------
    # Retry loop
    # ----------------------------------------------------------------
    for attempt_number in range(1, max_retries + 1):

        print_and_log(
            f"\n{'='*50}\nATTEMPT {attempt_number} of {max_retries}\n{'='*50}",
            session_log_filepath,
        )

        # --- Call the API ---
        # Note: ask_mistral_api() raises on non-200 — let it propagate.
        response = ask_mistral_api(context_history, use_this_model)

        # --- Extract raw reply string from response object ---
        try:
            response_data = response.json()
            raw_reply_string = response_data["choices"][0]["message"]["content"]

        except Exception as e:
            print_and_log(
                (
                    f"\nATTEMPT {attempt_number}: Could not extract reply string"
                    f" from response object.\n{e}\n{traceback.format_exc()}"
                ),
                session_log_filepath,
            )
            continue

        # --- Print and log the complete raw API reply ---
        print_and_log(
            f"\nRAW API RESPONSE (attempt {attempt_number}):\n{raw_reply_string}",
            session_log_filepath,
        )

        # --- Attempt JSON extraction and schema key validation ---
        print_and_log(
            f"\nAttempting JSON extraction and validation (attempt {attempt_number})...",
            session_log_filepath,
        )

        try:
            validated_dict = extract_json_as_pydict_from_unstructured_text(
                raw_reply_string,
                target_schema_model_dict,
            )

        except Exception as e:
            print_and_log(
                (
                    f"\nATTEMPT {attempt_number}: extract_json_as_pydict raised"
                    f" an exception.\n{e}\n{traceback.format_exc()}"
                ),
                session_log_filepath,
            )
            continue

        # --- Check result: False means malformed or wrong keys ---
        if validated_dict is False:
            print_and_log(
                (
                    f"\nATTEMPT {attempt_number} FAILED:"
                    f" extraction returned False (malformed JSON or wrong keys)."
                    f" Retrying...\n"
                ),
                session_log_filepath,
            )
            continue

        # ------------------------------------------------------------
        # SUCCESS
        # ------------------------------------------------------------
        print_and_log(
            (
                f"\nATTEMPT {attempt_number} SUCCEEDED.\n"
                f"Extracted dict:\n{json.dumps(validated_dict, indent=2)}"
            ),
            session_log_filepath,
        )

        save_extraction_results(
            raw_text_blob=raw_text_blob,
            target_schema_model_dict=target_schema_model_dict,
            use_this_model=use_this_model,
            result_dict=validated_dict,
            attempts_made=attempt_number,
            session_log_filepath=session_log_filepath,
        )

        return validated_dict

    # ----------------------------------------------------------------
    # Loop exhausted — all attempts failed
    # ----------------------------------------------------------------
    print_and_log(
        (
            f"\nWARNING: All {max_retries} attempts exhausted."
            f" Extraction failed."
            f" Returning None.\n"
        ),
        session_log_filepath,
    )

    save_extraction_results(
        raw_text_blob=raw_text_blob,
        target_schema_model_dict=target_schema_model_dict,
        use_this_model=use_this_model,
        result_dict=None,
        attempts_made=max_retries,
        session_log_filepath=session_log_filepath,
    )

    return None

# Manual Run

In [ ]:
"""
Input Fields

def run_extraction_with_retry(
    raw_text_blob: str,
    target_schema_model_dict: dict,
    extraction_prompt_string: str,
    use_this_model: str,
    max_retries: int = 3,
)
"""

run_extraction_result = run_extraction_with_retry(
    raw_text_blob,
    target_schema_model_dict,
    extraction_prompt_string,
    use_this_model,
    max_retries=3,
)

print(f"run_extraction_result -> {run_extraction_result}")


SESSION LOG
Timestamp      : 20260308_154608_597791
Model          : open-mistral-7b
Schema         :
{
  "animal_type": "",
  "weight_kg": 0.0,
  "height_cm": 0.0,
  "age_years": 0,
  "number_of_friends": 0,
  "birth_date": "",
  "birth_unix": 0,
  "color": "",
  "can_fly": false,
  "can_swim": false,
  "can_run": false,
  "watches_youtube": false,
  "daily_food_grams": 0.0,
  "popularity_score": 0.0
}
Raw Input Blob :

This is a cat! It is 23 cm tall. It gets 426 grams of food each day and weighs 9 kg! It is orange, and it was born on 1923-09-23. It is 3 years old! It can't swim. It can_ fly. It likes to watch youtube. It has 5 friends and has a popularity_score of 9.



Context history constructed.
Starting extraction loop.
Max retries: 3


ATTEMPT 1 of 3

RAW API RESPONSE (attempt 1):
```json
{
  "animal_type": "cat",
  "weight_kg": 9.0,
  "height_cm": 23.0,
  "age_years": 3,
  "number_of_friends": 5,
  "birth_date": "1923-09-23",
  "birth_unix": 638249600,
  "color": "orange",
  

# Test API


In [ ]:
## Optional Test of api system / connection
simple_ask_mistral_cloud("Hellow world", use_this_model)

'Hello! 🌍✨\n\nHow can I assist you today? Whether you have a question, need help with something, or just want to chat, I\'m here for you! Let me know what\'s on your mind. 😊\n\n(And yes, "Hello world!" is a classic—it\'s how many programming languages start their journey!)'

# csv Row Extraction Evalution-Testing Code

In [ ]:
import csv
import random


def round_float_to_three_decimals(value_to_round: float) -> float:
    """
    Round a floating point number to exactly 3 decimal places.

    Intent:
    -------
    Float values in the source CSV often have many decimal places
    (e.g., 28.602669690571616). For human-readable blurbs, we want
    cleaner numbers (e.g., 28.603). Three decimal places provides
    sufficient precision while maintaining readability.

    Parameters:
    -----------
    value_to_round : float
        The floating point number to be rounded.

    Returns:
    --------
    float
        The input value rounded to 3 decimal places.

    Example:
    --------
    >>> round_float_to_three_decimals(28.602669690571616)
    28.603
    """
    rounded_value = round(value_to_round, 3)
    return rounded_value


def convert_boolean_string_to_python_bool(boolean_string: str) -> bool:
    """
    Convert a string representation of a boolean to a Python bool.

    Intent:
    -------
    CSV files store all values as strings. When we read "True" or "False"
    from a CSV, we get the string "True", not the Python boolean True.
    This function handles that conversion safely.

    Parameters:
    -----------
    boolean_string : str
        A string that should be either "True" or "False" (case-sensitive
        as produced by Python's str(bool) conversion).

    Returns:
    --------
    bool
        True if the string is "True", False otherwise.

    Notes:
    ------
    This function treats any value that is not exactly "True" as False.
    This is a deliberate safety choice: unknown values default to False
    rather than raising an exception, making the pipeline more robust
    to minor data variations.
    """
    if boolean_string == "True":
        return True
    else:
        return False


def convert_can_ability_bool_to_phrase(can_do_ability: bool) -> str:
    """
    Convert a boolean ability flag to a natural language phrase.

    Intent:
    -------
    For abilities like "can_fly", "can_swim", "can_run", we want to
    produce readable phrases like "can" or "cannot" for use in
    natural language blurbs.

    Parameters:
    -----------
    can_do_ability : bool
        True if the animal has this ability, False otherwise.

    Returns:
    --------
    str
        "can" if True, "cannot" if False.

    Example:
    --------
    >>> convert_can_ability_bool_to_phrase(True)
    "can"
    >>> convert_can_ability_bool_to_phrase(False)
    "cannot"
    """
    if can_do_ability:
        return "can"
    else:
        return "cannot"


def convert_watches_youtube_bool_to_phrase(watches_youtube: bool) -> str:
    """
    Convert the watches_youtube boolean to a natural language phrase.

    Intent:
    -------
    The "watches_youtube" field needs a slightly different phrasing
    than abilities. We use "likes to" / "does not like to" to make
    grammatically correct sentences like "It likes to watch youtube."

    Parameters:
    -----------
    watches_youtube : bool
        True if the animal watches youtube, False otherwise.

    Returns:
    --------
    str
        "likes to" if True, "does not like to" if False.

    Example:
    --------
    >>> convert_watches_youtube_bool_to_phrase(True)
    "likes to"
    >>> convert_watches_youtube_bool_to_phrase(False)
    "does not like to"
    """
    if watches_youtube:
        return "likes to"
    else:
        return "does not like to"


def select_template_index_for_row(row_index: int) -> int:
    """
    Deterministically select which template (0, 1, or 2) to use for a row.

    Intent:
    -------
    We want variety in the blurb formats (to make vector search more
    realistic/challenging), but we also want reproducibility (so tests
    are repeatable). Using modulo arithmetic on the row index achieves
    both goals: the template varies by row, but the same row always
    gets the same template.

    Parameters:
    -----------
    row_index : int
        The zero-based index of the current row in the dataset.

    Returns:
    --------
    int
        An integer 0, 1, or 2 indicating which template to use.

    Example:
    --------
    >>> select_template_index_for_row(0)
    0
    >>> select_template_index_for_row(1)
    1
    >>> select_template_index_for_row(2)
    2
    >>> select_template_index_for_row(3)
    0
    """
    template_index = row_index % 3
    return template_index


def generate_blurb_using_template_zero(
    animal_type: str,
    weight_kg_rounded: float,
    height_cm_rounded: float,
    age_years: int,
    number_of_friends: int,
    birth_date: str,
    color: str,
    can_fly_phrase: str,
    can_swim_phrase: str,
    can_run_phrase: str,
    watches_youtube_phrase: str,
    daily_food_grams_rounded: float,
    popularity_score_rounded: float
) -> str:
    """
    Generate an unstructured text blurb using template format 0.

    Intent:
    -------
    This is the first of three template formats. It presents information
    in a specific order starting with animal type, then food, age, friends,
    physical characteristics, and abilities.

    Parameters:
    -----------
    animal_type : str
        The type of animal (e.g., "cat", "dog", "bird").
    weight_kg_rounded : float
        The animal's weight in kg, rounded to 3 decimal places.
    height_cm_rounded : float
        The animal's height in cm, rounded to 3 decimal places.
    age_years : int
        The animal's age in years.
    number_of_friends : int
        How many friends the animal has.
    birth_date : str
        The animal's birth date as a string (e.g., "2021-01-25").
    color : str
        The animal's color.
    can_fly_phrase : str
        "can" or "cannot" for flying ability.
    can_swim_phrase : str
        "can" or "cannot" for swimming ability.
    can_run_phrase : str
        "can" or "cannot" for running ability.
    watches_youtube_phrase : str
        "likes to" or "does not like to" for youtube watching.
    daily_food_grams_rounded : float
        Daily food allowance in grams, rounded to 3 decimal places.
    popularity_score_rounded : float
        Popularity score, rounded to 3 decimal places.

    Returns:
    --------
    str
        A natural language blurb describing the animal.
    """
    blurb_text = (
        f"This is about a {animal_type}. "
        f"It gets a daily allowance of {daily_food_grams_rounded} grams of food. "
        f"It is {age_years} years old, and has {number_of_friends} friends. "
        f"This {animal_type} weighs {weight_kg_rounded} kg, and is {height_cm_rounded} cm tall. "
        f"It has a popularity_score of {popularity_score_rounded}. "
        f"It was born on {birth_date}, and is {color}. "
        f"It {can_fly_phrase} fly, and {can_swim_phrase} swim, and {can_run_phrase} run. "
        f"It {watches_youtube_phrase} watch youtube."
    )
    return blurb_text


def generate_blurb_using_template_one(
    animal_type: str,
    weight_kg_rounded: float,
    height_cm_rounded: float,
    age_years: int,
    number_of_friends: int,
    birth_date: str,
    color: str,
    can_fly_phrase: str,
    can_swim_phrase: str,
    can_run_phrase: str,
    watches_youtube_phrase: str,
    daily_food_grams_rounded: float,
    popularity_score_rounded: float
) -> str:
    """
    Generate an unstructured text blurb using template format 1.

    Intent:
    -------
    This is the second of three template formats. It presents information
    in a different order than template 0, and uses exclamation marks for
    some ability statements to create textual variety.

    Parameters:
    -----------
    (Same as generate_blurb_using_template_zero)

    Returns:
    --------
    str
        A natural language blurb describing the animal.
    """
    blurb_text = (
        f"This {animal_type} weighs {weight_kg_rounded} kg, and is {height_cm_rounded} cm tall. "
        f"It is {color} and was born on {birth_date}. "
        f"It gets {daily_food_grams_rounded} grams of food each day. "
        f"It is {age_years} years old. "
        f"It {can_fly_phrase} fly! "
        f"It {can_swim_phrase} swim! "
        f"It {can_run_phrase} run! "
        f"This {animal_type} has {number_of_friends} friends and has a popularity_score of {popularity_score_rounded}. "
        f"It {watches_youtube_phrase} watch youtube."
    )
    return blurb_text


def generate_blurb_using_template_two(
    animal_type: str,
    weight_kg_rounded: float,
    height_cm_rounded: float,
    age_years: int,
    number_of_friends: int,
    birth_date: str,
    color: str,
    can_fly_phrase: str,
    can_swim_phrase: str,
    can_run_phrase: str,
    watches_youtube_phrase: str,
    daily_food_grams_rounded: float,
    popularity_score_rounded: float
) -> str:
    """
    Generate an unstructured text blurb using template format 2.

    Intent:
    -------
    This is the third of three template formats. It presents information
    in yet another order, with different punctuation patterns (exclamation
    at the start, periods for abilities in the middle).

    Parameters:
    -----------
    (Same as generate_blurb_using_template_zero)

    Returns:
    --------
    str
        A natural language blurb describing the animal.
    """
    blurb_text = (
        f"This is a {animal_type}! "
        f"It is {height_cm_rounded} cm tall. "
        f"It gets {daily_food_grams_rounded} grams of food each day and weighs {weight_kg_rounded} kg! "
        f"It is {color}, and it was born on {birth_date}. "
        f"It {can_run_phrase} run. "
        f"It is {age_years} years old! "
        f"It {can_swim_phrase} swim. "
        f"It {can_fly_phrase} fly. "
        f"It {watches_youtube_phrase} watch youtube. "
        f"It has {number_of_friends} friends and has a popularity_score of {popularity_score_rounded}."
    )
    return blurb_text


def generate_blurb_for_row(row_data: dict, row_index: int) -> str:
    """
    Generate an unstructured text blurb for a single row of data.

    Intent:
    -------
    This function is the main orchestrator for blurb generation. It:
    1. Extracts and converts all necessary fields from the row
    2. Rounds float values to 3 decimal places
    3. Converts boolean strings to natural language phrases
    4. Selects the appropriate template based on row index
    5. Calls the appropriate template function

    This function encapsulates all the data transformation logic,
    keeping the main processing loop clean and simple.

    Parameters:
    -----------
    row_data : dict
        A dictionary containing one row of CSV data, where keys are
        column names and values are string representations of the data.
    row_index : int
        The zero-based index of this row in the dataset.

    Returns:
    --------
    str
        A natural language blurb describing the animal in this row.

    Raises:
    -------
    ValueError
        If required fields are missing or cannot be converted.
    KeyError
        If expected column names are not present in row_data.
    """
    # Extract and convert string values to appropriate types
    # Note: CSV reader returns all values as strings

    animal_type = row_data["animal_type"]
    color = row_data["color"]
    birth_date = row_data["birth_date"]

    # Convert numeric strings to numbers, then round floats
    weight_kg_raw = float(row_data["weight_kg"])
    weight_kg_rounded = round_float_to_three_decimals(weight_kg_raw)

    height_cm_raw = float(row_data["height_cm"])
    height_cm_rounded = round_float_to_three_decimals(height_cm_raw)

    daily_food_grams_raw = float(row_data["daily_food_grams"])
    daily_food_grams_rounded = round_float_to_three_decimals(daily_food_grams_raw)

    popularity_score_raw = float(row_data["popularity_score"])
    popularity_score_rounded = round_float_to_three_decimals(popularity_score_raw)

    # Integer fields
    age_years = int(row_data["age_years"])
    number_of_friends = int(row_data["number_of_friends"])

    # Convert boolean strings to Python bools, then to phrases
    can_fly_bool = convert_boolean_string_to_python_bool(row_data["can_fly"])
    can_fly_phrase = convert_can_ability_bool_to_phrase(can_fly_bool)

    can_swim_bool = convert_boolean_string_to_python_bool(row_data["can_swim"])
    can_swim_phrase = convert_can_ability_bool_to_phrase(can_swim_bool)

    can_run_bool = convert_boolean_string_to_python_bool(row_data["can_run"])
    can_run_phrase = convert_can_ability_bool_to_phrase(can_run_bool)

    watches_youtube_bool = convert_boolean_string_to_python_bool(row_data["watches_youtube"])
    watches_youtube_phrase = convert_watches_youtube_bool_to_phrase(watches_youtube_bool)

    # Select template based on row index
    template_index = select_template_index_for_row(row_index)

    # Generate blurb using the selected template
    if template_index == 0:
        generated_blurb = generate_blurb_using_template_zero(
            animal_type=animal_type,
            weight_kg_rounded=weight_kg_rounded,
            height_cm_rounded=height_cm_rounded,
            age_years=age_years,
            number_of_friends=number_of_friends,
            birth_date=birth_date,
            color=color,
            can_fly_phrase=can_fly_phrase,
            can_swim_phrase=can_swim_phrase,
            can_run_phrase=can_run_phrase,
            watches_youtube_phrase=watches_youtube_phrase,
            daily_food_grams_rounded=daily_food_grams_rounded,
            popularity_score_rounded=popularity_score_rounded
        )
    elif template_index == 1:
        generated_blurb = generate_blurb_using_template_one(
            animal_type=animal_type,
            weight_kg_rounded=weight_kg_rounded,
            height_cm_rounded=height_cm_rounded,
            age_years=age_years,
            number_of_friends=number_of_friends,
            birth_date=birth_date,
            color=color,
            can_fly_phrase=can_fly_phrase,
            can_swim_phrase=can_swim_phrase,
            can_run_phrase=can_run_phrase,
            watches_youtube_phrase=watches_youtube_phrase,
            daily_food_grams_rounded=daily_food_grams_rounded,
            popularity_score_rounded=popularity_score_rounded
        )
    else:
        generated_blurb = generate_blurb_using_template_two(
            animal_type=animal_type,
            weight_kg_rounded=weight_kg_rounded,
            height_cm_rounded=height_cm_rounded,
            age_years=age_years,
            number_of_friends=number_of_friends,
            birth_date=birth_date,
            color=color,
            can_fly_phrase=can_fly_phrase,
            can_swim_phrase=can_swim_phrase,
            can_run_phrase=can_run_phrase,
            watches_youtube_phrase=watches_youtube_phrase,
            daily_food_grams_rounded=daily_food_grams_rounded,
            popularity_score_rounded=popularity_score_rounded
        )

    return generated_blurb


def read_csv_file_to_list_of_dicts(input_file_path: str) -> tuple:
    """
    Read a CSV file and return its contents as a list of dictionaries.

    Intent:
    -------
    This function handles the file I/O for reading the input CSV.
    It uses csv.DictReader which automatically uses the first row
    as column headers and returns each subsequent row as a dictionary.

    We also return the fieldnames separately so we can preserve
    column order when writing the output file.

    Parameters:
    -----------
    input_file_path : str
        The path to the input CSV file.

    Returns:
    --------
    tuple(list[dict], list[str])
        A tuple containing:
        - A list of dictionaries, one per row
        - A list of field names (column headers) in original order

    Raises:
    -------
    FileNotFoundError
        If the input file does not exist.
    PermissionError
        If the file cannot be read due to permissions.
    ValueError
        If the CSV file is empty or has no header row.
    """
    list_of_row_dicts = []
    fieldnames_list = []

    with open(input_file_path, mode='r', encoding='utf-8', newline='') as csv_file_handle:
        csv_dict_reader = csv.DictReader(csv_file_handle)

        # Read all rows into memory as dictionaries
        for row_dict in csv_dict_reader:
            list_of_row_dicts.append(row_dict)

        # Capture the fieldnames from the reader AFTER reading rows
        reader_fieldnames = csv_dict_reader.fieldnames

        if reader_fieldnames is None:
            raise ValueError(
                f"CSV file appears to be empty or has no header row: {input_file_path}"
            )

        fieldnames_list = list(reader_fieldnames)

    return list_of_row_dicts, fieldnames_list

In [ ]:
# =============================================================================
# SECTION 3: GROUND TRUTH PREPARATION
# =============================================================================
# Converts raw CSV row dicts into canonical ground-truth dicts that match
# the extraction schema. Values are typed and rounded to match what the
# blurb contains, so comparison is apples-to-apples.
# =============================================================================


def prepare_ground_truth_row(row_data: dict) -> dict:
    """
    Convert a raw CSV row dict into a ground-truth dict matching the
    extraction schema.

    Intent:
    -------
    The CSV reader returns all values as strings. The extraction pipeline
    returns typed values (int, float, bool, str). For comparison to be
    meaningful, we need the ground truth in the same typed format.

    Float values are rounded to 3 decimal places to match what appears
    in the blurb text (the blurb generator rounds to 3 decimals).

    The birth_unix field is included from the CSV even though it does
    NOT appear in the blurb text. If the model cannot infer it, that
    will be recorded as a legitimate extraction error.

    Parameters:
    -----------
    row_data : dict
        A dictionary from csv.DictReader, where all values are strings.
        Expected to contain all keys in the extraction schema.

    Returns:
    --------
    dict
        A dictionary with keys matching target_schema_model_dict and
        values cast to their correct Python types. Floats are rounded
        to 3 decimal places.

    Raises:
    -------
    KeyError
        If expected fields are missing from row_data.
    ValueError
        If values cannot be converted to expected types.
    """
    ground_truth_dict = {
        # String fields (no conversion needed beyond str)
        "animal_type": str(row_data["animal_type"]),
        "color": str(row_data["color"]),

        # Date string field (keep as-is, already YYYY-MM-DD in CSV)
        "birth_date": str(row_data["birth_date"]),

        # Integer fields
        "age_years": int(row_data["age_years"]),
        "number_of_friends": int(row_data["number_of_friends"]),
        "birth_unix": int(row_data["birth_unix"]),

        # Float fields (rounded to 3 decimals to match blurb content)
        "weight_kg": round_float_to_three_decimals(float(row_data["weight_kg"])),
        "height_cm": round_float_to_three_decimals(float(row_data["height_cm"])),
        "daily_food_grams": round_float_to_three_decimals(float(row_data["daily_food_grams"])),
        "popularity_score": round_float_to_three_decimals(float(row_data["popularity_score"])),

        # Boolean fields (CSV stores as "True"/"False" strings)
        "can_fly": convert_boolean_string_to_python_bool(row_data["can_fly"]),
        "can_swim": convert_boolean_string_to_python_bool(row_data["can_swim"]),
        "can_run": convert_boolean_string_to_python_bool(row_data["can_run"]),
        "watches_youtube": convert_boolean_string_to_python_bool(row_data["watches_youtube"]),
    }

    return ground_truth_dict

In [ ]:
# =============================================================================
# SECTION 4: FIELD-LEVEL COMPARISON
# =============================================================================
# Functions to compare extracted dicts against ground truth, field by field.
# =============================================================================


def compare_single_field(
    field_name: str,
    expected_value,
    extracted_value,
) -> bool:
    """
    Compare a single field value between ground truth and extraction result.

    Intent:
    -------
    This performs exact equality comparison after normalizing both values
    to the same type. This is the atomic unit of comparison — every
    scoring metric ultimately depends on this function.

    Both values are compared after being cast to their schema type.
    If either value is None, it only matches if both are None.

    No fuzzy matching, no rounding tolerance — exact match only.
    (Rounding tolerance is deferred to a future scope if ever needed.)

    Parameters:
    -----------
    field_name : str
        The name of the field being compared (for debugging context).
    expected_value : any
        The ground-truth value (already typed by prepare_ground_truth_row).
    extracted_value : any
        The value from the extraction result (already typed by the
        extraction pipeline's validation/coercion).

    Returns:
    --------
    bool
        True if the values are exactly equal, False otherwise.
    """
    # Handle None cases
    if expected_value is None and extracted_value is None:
        return True
    if expected_value is None or extracted_value is None:
        return False

    # Direct equality comparison
    # Both values should already be in their correct types from
    # prepare_ground_truth_row() and validate_and_coerce_extracted_dict()
    return expected_value == extracted_value


def compare_row_to_ground_truth(
    extracted_dict: dict,
    ground_truth_dict: dict,
) -> dict:
    """
    Compare all fields between an extracted dict and its ground truth.

    Intent:
    -------
    This is the per-row comparison function. It iterates over every field
    in the ground truth, compares to the extracted value, and produces
    a structured result that can be aggregated for scoring.

    If extracted_dict is None (extraction failed entirely), all fields
    are marked as mismatched with extracted value of None.

    Parameters:
    -----------
    extracted_dict : dict or None
        The dictionary returned by the extraction pipeline.
        None if extraction failed completely.
    ground_truth_dict : dict
        The ground-truth dictionary from prepare_ground_truth_row().

    Returns:
    --------
    dict
        A result dictionary with the following structure:
        {
            "row_match": bool,
                True if ALL fields matched, False otherwise.

            "extraction_failed": bool,
                True if extracted_dict was None (total failure).

            "field_results": {
                "field_name": True/False,
                ...
            },
                Per-field match results.

            "mismatched_fields": {
                "field_name": {
                    "expected": <ground_truth_value>,
                    "got": <extracted_value>
                },
                ...
            },
                Details for fields that did NOT match.
                Only populated for mismatched fields.

            "fields_correct_count": int,
                Number of fields that matched.

            "fields_total_count": int,
                Total number of fields compared.
        }
    """
    extraction_failed = (extracted_dict is None)

    field_results = {}
    mismatched_fields = {}
    fields_correct_count = 0
    fields_total_count = 0

    for field_name, expected_value in ground_truth_dict.items():

        fields_total_count += 1

        # If extraction failed entirely, every field is a mismatch
        if extraction_failed:
            extracted_value = None
        else:
            extracted_value = extracted_dict.get(field_name, None)

        is_match = compare_single_field(
            field_name=field_name,
            expected_value=expected_value,
            extracted_value=extracted_value,
        )

        field_results[field_name] = is_match

        if is_match:
            fields_correct_count += 1
        else:
            mismatched_fields[field_name] = {
                "expected": expected_value,
                "got": extracted_value,
            }

    row_match = (fields_correct_count == fields_total_count)

    comparison_result = {
        "row_match": row_match,
        "extraction_failed": extraction_failed,
        "field_results": field_results,
        "mismatched_fields": mismatched_fields,
        "fields_correct_count": fields_correct_count,
        "fields_total_count": fields_total_count,
    }

    return comparison_result

# Test Loop Code

In [ ]:
# =============================================================================
# SECTION 5: EVAL LOOP
# =============================================================================
# The main evaluation loop that ties together:
#   CSV reading -> blurb generation -> extraction -> comparison
# =============================================================================


def run_row_extraction_eval(
    csv_path: str,
    n_rows: int,
    offset: int,
    model: str,
    schema_dict: dict,
    schema_doc: str,
    field_type_schema: dict,
    max_retries: int,
    random_seed: int,
) -> list:
    """
    Run the full row-by-row extraction evaluation pipeline.

    Intent:
    -------
    This is the main orchestration function for Mode 1 evaluation.
    For each row in the specified range:
    1. Generate a blurb from the structured data
    2. Build an extraction prompt from the blurb
    3. Run the extraction pipeline (API call with retries)
    4. Compare extracted result to ground truth
    5. Collect all results for scoring

    The random seed is set once at the start for reproducibility.

    Parameters:
    -----------
    csv_path : str
        Path to the source CSV file.
    n_rows : int
        Number of rows to evaluate.
    offset : int
        Starting row index (0-based) in the CSV.
    model : str
        The Mistral model name string to use for extraction.
    schema_dict : dict
        The target schema dictionary (target_schema_model_dict).
    schema_doc : str
        The data schema documentation blurb (data_schema_doc_blurb).
    field_type_schema : dict
        The field type schema for validation (FIELD_TYPE_SCHEMA).
    max_retries : int
        Maximum extraction retries per row.
    random_seed : int
        Random seed for reproducibility.

    Returns:
    --------
    list[dict]
        A list of per-row result dictionaries, each containing:
        {
            "row_index": int,
                The original row index in the CSV.

            "blurb": str,
                The generated blurb text.

            "ground_truth": dict,
                The typed ground-truth dictionary.

            "extracted": dict or None,
                The extraction result (None if extraction failed).

            "comparison": dict,
                The output of compare_row_to_ground_truth().
        }
    """
    # --- Set random seed for reproducibility ---
    random.seed(random_seed)

    # --- Read CSV ---
    print(f"Reading CSV: {csv_path}")
    all_rows, fieldnames = read_csv_file_to_list_of_dicts(csv_path)
    total_rows_available = len(all_rows)
    print(f"Total rows in CSV: {total_rows_available}")

    # --- Validate offset and n_rows ---
    if offset >= total_rows_available:
        raise ValueError(
            f"Offset ({offset}) is >= total rows available ({total_rows_available})."
        )

    end_index = min(offset + n_rows, total_rows_available)
    actual_n_rows = end_index - offset

    if actual_n_rows < n_rows:
        print(
            f"WARNING: Requested {n_rows} rows but only {actual_n_rows} "
            f"available from offset {offset}. Evaluating {actual_n_rows} rows."
        )

    # --- Slice rows ---
    eval_rows = all_rows[offset:end_index]

    print(f"\nStarting evaluation: {actual_n_rows} rows, offset={offset}, model={model}")
    print(f"Random seed: {random_seed}")
    print(f"=" * 60)

    # --- Main eval loop ---
    eval_results = []

    for i, row_data in enumerate(eval_rows):

        # The original CSV row index (for template selection and reporting)
        original_row_index = offset + i

        print(f"\n{'='*60}")
        print(f"EVAL ROW {i+1} of {actual_n_rows}  (CSV row index: {original_row_index})")
        print(f"{'='*60}")

        # --- Step 1: Generate blurb ---
        blurb = generate_blurb_for_row(row_data, original_row_index)
        print(f"\nGenerated blurb:\n{blurb}")

        # --- Step 2: Build extraction prompt ---
        extraction_prompt = build_extraction_prompt(
            raw_text_input=blurb,
            data_schema=schema_dict,
            description_of_data_schema=schema_doc,
        )

        # --- Step 3: Run extraction ---
        extracted_dict = run_extraction_with_retry(
            raw_text_blob=blurb,
            target_schema_model_dict=schema_dict,
            extraction_prompt_string=extraction_prompt,
            use_this_model=model,
            max_retries=max_retries,
        )

        # --- Step 4: Prepare ground truth ---
        ground_truth = prepare_ground_truth_row(row_data)

        # --- Step 5: Compare ---
        comparison = compare_row_to_ground_truth(
            extracted_dict=extracted_dict,
            ground_truth_dict=ground_truth,
        )

        # --- Log this row's results ---
        print(f"\nGround truth: {json.dumps(ground_truth, indent=2, default=str)}")
        print(f"\nExtracted:    {json.dumps(extracted_dict, indent=2, default=str) if extracted_dict else 'None (extraction failed)'}")
        print(f"\nRow match: {comparison['row_match']}")
        print(f"Fields correct: {comparison['fields_correct_count']}/{comparison['fields_total_count']}")

        if comparison['mismatched_fields']:
            print(f"Mismatched fields:")
            for field_name, mismatch_detail in comparison['mismatched_fields'].items():
                print(f"  {field_name}: expected={mismatch_detail['expected']}, got={mismatch_detail['got']}")

        # --- Store result ---
        row_result = {
            "row_index": original_row_index,
            "blurb": blurb,
            "ground_truth": ground_truth,
            "extracted": extracted_dict,
            "comparison": comparison,
        }
        eval_results.append(row_result)

    print(f"\n{'='*60}")
    print(f"Evaluation loop complete. {len(eval_results)} rows evaluated.")
    print(f"{'='*60}")

    return eval_results

# Scoring Reporting Code

In [ ]:
# =============================================================================
# SECTION 6: SCORING AND REPORTING
# =============================================================================
# Aggregates per-row results into summary statistics and prints a report.
# =============================================================================


def compute_eval_summary(eval_results: list) -> dict:
    """
    Aggregate per-row evaluation results into summary statistics.

    Intent:
    -------
    This takes the list of per-row result dicts from run_row_extraction_eval()
    and computes aggregate metrics that characterize the extraction pipeline's
    performance over the full evaluation run.

    Parameters:
    -----------
    eval_results : list[dict]
        The list of per-row result dicts as returned by run_row_extraction_eval().

    Returns:
    --------
    dict
        A summary dictionary containing:
        {
            "total_rows_evaluated": int,

            "total_extraction_failures": int,
                Count of rows where extraction returned None.

            "total_rows_all_fields_correct": int,
                Count of rows where every field matched.

            "overall_row_accuracy_pct": float,
                Percentage of rows with all fields correct.

            "per_field_stats": {
                "field_name": {
                    "correct_count": int,
                    "incorrect_count": int,
                    "accuracy_pct": float,
                },
                ...
            },

            "total_field_errors": int,
                Sum of all field-level errors across all rows.

            "per_field_error_share_pct": {
                "field_name": float,
                    What percentage of total errors this field represents.
            },

            "most_common_errors": list[tuple],
                Fields sorted by error count descending.
                Each element: (field_name, error_count, error_share_pct)
        }
    """
    total_rows = len(eval_results)

    if total_rows == 0:
        return {
            "total_rows_evaluated": 0,
            "total_extraction_failures": 0,
            "total_rows_all_fields_correct": 0,
            "overall_row_accuracy_pct": 0.0,
            "per_field_stats": {},
            "total_field_errors": 0,
            "per_field_error_share_pct": {},
            "most_common_errors": [],
        }

    # --- Count extraction failures ---
    total_extraction_failures = sum(
        1 for r in eval_results if r["comparison"]["extraction_failed"]
    )

    # --- Count perfect rows ---
    total_rows_all_fields_correct = sum(
        1 for r in eval_results if r["comparison"]["row_match"]
    )

    # --- Overall row accuracy ---
    overall_row_accuracy_pct = (total_rows_all_fields_correct / total_rows) * 100.0

    # --- Per-field stats ---
    # Initialize counters using the field names from the first result
    first_field_results = eval_results[0]["comparison"]["field_results"]
    field_names = list(first_field_results.keys())

    per_field_correct = {fn: 0 for fn in field_names}
    per_field_incorrect = {fn: 0 for fn in field_names}

    for row_result in eval_results:
        field_results = row_result["comparison"]["field_results"]
        for fn in field_names:
            if field_results.get(fn, False):
                per_field_correct[fn] += 1
            else:
                per_field_incorrect[fn] += 1

    per_field_stats = {}
    for fn in field_names:
        correct = per_field_correct[fn]
        incorrect = per_field_incorrect[fn]
        accuracy = (correct / total_rows) * 100.0 if total_rows > 0 else 0.0
        per_field_stats[fn] = {
            "correct_count": correct,
            "incorrect_count": incorrect,
            "accuracy_pct": accuracy,
        }

    # --- Total field errors ---
    total_field_errors = sum(per_field_incorrect[fn] for fn in field_names)

    # --- Per-field error share ---
    per_field_error_share_pct = {}
    for fn in field_names:
        if total_field_errors > 0:
            share = (per_field_incorrect[fn] / total_field_errors) * 100.0
        else:
            share = 0.0
        per_field_error_share_pct[fn] = share

    # --- Most common errors (sorted descending by error count) ---
    most_common_errors = sorted(
        [
            (fn, per_field_incorrect[fn], per_field_error_share_pct[fn])
            for fn in field_names
        ],
        key=lambda x: x[1],
        reverse=True,
    )

    summary = {
        "total_rows_evaluated": total_rows,
        "total_extraction_failures": total_extraction_failures,
        "total_rows_all_fields_correct": total_rows_all_fields_correct,
        "overall_row_accuracy_pct": overall_row_accuracy_pct,
        "per_field_stats": per_field_stats,
        "total_field_errors": total_field_errors,
        "per_field_error_share_pct": per_field_error_share_pct,
        "most_common_errors": most_common_errors,
    }

    return summary

In [ ]:
def print_eval_report(eval_summary: dict) -> None:
    """
    Pretty-print the evaluation summary report.

    Intent:
    -------
    Produces a human-readable report of the evaluation results,
    designed to fit comfortably in a notebook output cell or terminal.
    Fields are sorted by error rate so the worst-performing fields
    appear first, making it easy to identify problem areas.

    Parameters:
    -----------
    eval_summary : dict
        The summary dictionary from compute_eval_summary().

    Returns:
    --------
    None
        Prints directly to stdout.
    """
    print("\n")
    print("=" * 70)
    print("  ROW-LEVEL EXTRACTION EVALUATION REPORT")
    print("=" * 70)

    # --- Overall stats ---
    print(f"\n  Total rows evaluated:           {eval_summary['total_rows_evaluated']}")
    print(f"  Extraction failures (None):     {eval_summary['total_extraction_failures']}")
    print(f"  Rows with all fields correct:   {eval_summary['total_rows_all_fields_correct']}")
    print(f"  Overall row accuracy:           {eval_summary['overall_row_accuracy_pct']:.1f}%")
    print(f"  Total field-level errors:       {eval_summary['total_field_errors']}")

    # --- Per-field accuracy table ---
    print(f"\n  {'─' * 66}")
    print(f"  PER-FIELD ACCURACY")
    print(f"  {'─' * 66}")
    print(f"  {'Field':<25} {'Correct':>8} {'Errors':>8} {'Accuracy':>10} {'Err Share':>10}")
    print(f"  {'─' * 66}")

    # Sort by accuracy ascending (worst first)
    per_field_stats = eval_summary["per_field_stats"]
    error_shares = eval_summary["per_field_error_share_pct"]

    sorted_fields = sorted(
        per_field_stats.keys(),
        key=lambda fn: per_field_stats[fn]["accuracy_pct"],
    )

    for fn in sorted_fields:
        stats = per_field_stats[fn]
        share = error_shares.get(fn, 0.0)
        print(
            f"  {fn:<25} "
            f"{stats['correct_count']:>8} "
            f"{stats['incorrect_count']:>8} "
            f"{stats['accuracy_pct']:>9.1f}% "
            f"{share:>9.1f}%"
        )

    print(f"  {'─' * 66}")

    # --- Top error fields ---
    most_common = eval_summary["most_common_errors"]
    # Filter to only fields that actually had errors
    fields_with_errors = [(fn, cnt, share) for fn, cnt, share in most_common if cnt > 0]

    if fields_with_errors:
        print(f"\n  TOP ERROR FIELDS (descending by error count):")
        for rank, (fn, cnt, share) in enumerate(fields_with_errors, start=1):
            print(f"    {rank}. {fn}: {cnt} errors ({share:.1f}% of all errors)")
    else:
        print(f"\n  NO FIELD ERRORS — perfect extraction across all rows!")

    print(f"\n{'=' * 70}")
    print()

# Run Test (single row extraction)

In [ ]:
# =============================================================================
# SECTION 7: RUN EVALUATION
# =============================================================================
# Execute the evaluation with the configured parameters.
# =============================================================================

eval_results = run_row_extraction_eval(
    csv_path=EVAL_CSV_PATH,
    n_rows=EVAL_N_ROWS,
    offset=EVAL_OFFSET,
    model=use_this_model,
    schema_dict=target_schema_model_dict,
    schema_doc=data_schema_doc_blurb,
    field_type_schema=FIELD_TYPE_SCHEMA,
    max_retries=EVAL_MAX_RETRIES,
    random_seed=EVAL_RANDOM_SEED,
)

eval_summary = compute_eval_summary(eval_results)
print_eval_report(eval_summary)

Reading CSV: synthetic_biology_dataset.csv
Total rows in CSV: 20000

Starting evaluation: 2 rows, offset=0, model=open-mistral-7b
Random seed: 42

EVAL ROW 1 of 2  (CSV row index: 0)

Generated blurb:
This is about a bird. It gets a daily allowance of 85.277 grams of food. It is 9 years old, and has 7 friends. This bird weighs 28.603 kg, and is 60.916 cm tall. It has a popularity_score of 10.0. It was born on 2017-12-04, and is gray. It cannot fly, and can swim, and can run. It does not like to watch youtube.

SESSION LOG
Timestamp      : 20260308_154610_687265
Model          : open-mistral-7b
Schema         :
{
  "animal_type": "",
  "weight_kg": 0.0,
  "height_cm": 0.0,
  "age_years": 0,
  "number_of_friends": 0,
  "birth_date": "",
  "birth_unix": 0,
  "color": "",
  "can_fly": false,
  "can_swim": false,
  "can_run": false,
  "watches_youtube": false,
  "daily_food_grams": 0.0,
  "popularity_score": 0.0
}
Raw Input Blob :
This is about a bird. It gets a daily allowance of 85.277 gr

In [ ]:
# =============================================================================
# SECTION 8: DETAILED RESULTS INSPECTION (OPTIONAL)
# =============================================================================
# Inspect individual row results for debugging or detailed analysis.
# This cell can be run manually after the eval to drill into specifics.
# =============================================================================

# Print detailed mismatch report for every row that had errors
print("DETAILED MISMATCH REPORT")
print("=" * 70)

for row_result in eval_results:
    comparison = row_result["comparison"]

    # Skip perfect rows
    if comparison["row_match"]:
        continue

    row_idx = row_result["row_index"]
    print(f"\n--- CSV Row Index: {row_idx} ---")
    print(f"  Extraction failed entirely: {comparison['extraction_failed']}")
    print(f"  Fields correct: {comparison['fields_correct_count']}/{comparison['fields_total_count']}")

    if comparison["mismatched_fields"]:
        print(f"  Mismatches:")
        for field_name, detail in comparison["mismatched_fields"].items():
            print(f"    {field_name}:")
            print(f"      expected : {detail['expected']}")
            print(f"      got      : {detail['got']}")

    print(f"  Blurb: {row_result['blurb'][:120]}...")

print(f"\n{'='*70}")
print("End of detailed mismatch report.")

DETAILED MISMATCH REPORT

--- CSV Row Index: 0 ---
  Extraction failed entirely: False
  Fields correct: 13/14
  Mismatches:
    birth_unix:
      expected : 1512363600
      got      : 1512380800
  Blurb: This is about a bird. It gets a daily allowance of 85.277 grams of food. It is 9 years old, and has 7 friends. This bird...

--- CSV Row Index: 1 ---
  Extraction failed entirely: False
  Fields correct: 13/14
  Mismatches:
    birth_unix:
      expected : 1611550800
      got      : 1611420800
  Blurb: This cat weighs 2.769 kg, and is 95.216 cm tall. It is red and was born on 2021-01-25. It gets 119.217 grams of food eac...

End of detailed mismatch report.
